In [2]:
import numpy as np
np.set_printoptions(suppress=True, precision=6)
import pandas as pd
import matplotlib.pyplot as plt
import shap

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor


#LOAD/PREPROCESS DATA
data = pd.read_csv("vaData.csv")
# data = pd.read_csv("dftDataCuda2.csv")

X = data.drop('GPU', axis=1)
y = data['GPU']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
y_log = np.log(y_train)

# scaler = StandardScaler()
# X_train = scaler.fit_transform(X_train)
# X_test = scaler.fit_transform(X_test)

cols = ['total global memory(KB)', 'clock rate(KHz)', 'multiprocessor count',
       'async engine count', 'memory bus width', 'memory clock rate (KHz)',
       'L2 cache size (bytes)', 'max threads per SM']

features = ['elements', 'blocks', 'threadsPerBlock',
       'total global memory(KB)', 'clock rate(KHz)', 'multiprocessor count',
       'async engine count', 'memory bus width', 'memory clock rate (KHz)',
       'L2 cache size (bytes)', 'max threads per SM']

device_specs = data[cols].drop_duplicates().to_numpy()

#device_specs[0] = cuda2
#device_specs[1] = cuda3
#device_specs[2] = cuda4
#device_specs[3] = cuda5

In [3]:
# model = XGBRegressor()
model = XGBRegressor(
    n_estimators=10000,
    max_depth = 8,
    learning_rate=0.1,
    random_state=42)
model.fit(X_train, y_log) #train on y_log instead of y_train to restrict the predicted compute times to be >0

preds_log = model.predict(X_test)
preds = np.exp(preds_log) #revert back from log of predicted time
preds[:6], y_test.to_numpy()[:6]

(array([0.027138, 0.000261, 0.004528, 0.013934, 0.000853, 0.000499],
       dtype=float32),
 array([0.027308, 0.000267, 0.004199, 0.01362 , 0.000951, 0.000675]))

In [4]:
blocks = np.array([32, 64, 128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768])
threads = np.array([32, 50, 64, 100, 128, 200, 256, 400, 512, 750, 1024])
num_elements = 10000000

results = []
for b in blocks:
    for t in threads:

        #concatenate [elements, blocks, TPB] with rest of inputs (8,)
        #try with device_specs[x], x \in {0, 1, 2, 3}
        input_features = np.concatenate([np.array([num_elements, b, t], dtype='float64'), device_specs[2]]).reshape(-1, 11)
        
        #model was trained on log values, need to exponentiate once prediction is made
        res = np.exp(model.predict(input_features))

        results.append([b, t, res[0]])

results = np.array(results)
sorted = results[results[:, 2].argsort()] #sort array to determine the number of blocks and TPB with lowest runtimes 

min_b, min_tpb, min_time = results[np.argmin(results[:, 2])]
print(f'Best Config:')
print(f'Blocks: {int(min_b)} | Threads Per Block: {int(min_tpb)} | Time: {min_time:.6f}s')

print("\nBest device: cuda3")
print("Device Specs:\n")

for name, spec in zip(cols, device_specs[2]):
    print(f'{name}: {spec}')

Best Config:
Blocks: 64 | Threads Per Block: 200 | Time: 0.026472s

Best device: cuda3
Device Specs:

total global memory(KB): 12492800
clock rate(KHz): 1215500
multiprocessor count: 24
async engine count: 2
memory bus width: 384
memory clock rate (KHz): 3505000
L2 cache size (bytes): 3145728
max threads per SM: 2048


In [5]:
print('Top 10 Best Configurations:')

for s in range(10):
    nb, tpb, time = sorted[s]
    print(f'Blocks: {int(nb)} | Threads Per Block: {int(tpb)} | Time: {time:.6f}s')

#Compare with test set
sorted_df = data[(data['elements'] == num_elements)].sort_values(by='GPU')
sorted_df.head(10)

Top 10 Best Configurations:
Blocks: 64 | Threads Per Block: 200 | Time: 0.026472s
Blocks: 512 | Threads Per Block: 64 | Time: 0.026503s
Blocks: 4096 | Threads Per Block: 128 | Time: 0.026779s
Blocks: 512 | Threads Per Block: 256 | Time: 0.026790s
Blocks: 1024 | Threads Per Block: 50 | Time: 0.026793s
Blocks: 128 | Threads Per Block: 200 | Time: 0.026813s
Blocks: 4096 | Threads Per Block: 750 | Time: 0.026844s
Blocks: 2048 | Threads Per Block: 256 | Time: 0.026852s
Blocks: 64 | Threads Per Block: 100 | Time: 0.026872s
Blocks: 8192 | Threads Per Block: 64 | Time: 0.026912s


,elements,blocks,threadsPerBlock,GPU,total global memory(KB),clock rate(KHz),multiprocessor count,async engine count,memory bus width,memory clock rate (KHz),L2 cache size (bytes),max threads per SM
792,10000000,2048,32,0.012584,11268608,1635000,68,3,352,7000000,5767168,1024
772,10000000,512,64,0.012663,11268608,1635000,68,3,352,7000000,5767168,1024
829,10000000,16384,128,0.012693,11268608,1635000,68,3,352,7000000,5767168,1024
794,10000000,2048,64,0.012695,11268608,1635000,68,3,352,7000000,5767168,1024
753,10000000,128,200,0.012700,11268608,1635000,68,3,352,7000000,5767168,1024
830,10000000,16384,200,0.012711,11268608,1635000,68,3,352,7000000,5767168,1024
752,10000000,128,128,0.012724,11268608,1635000,68,3,352,7000000,5767168,1024
809,10000000,4096,256,0.012728,11268608,1635000,68,3,352,7000000,5767168,1024
810,10000000,4096,400,0.012734,11268608,1635000,68,3,352,7000000,5767168,1024
754,10000000,128,256,0.012745,11268608,1635000,68,3,352,7000000,5767168,1024


In [7]:
#Compare with Ground Truth for 100k Elements
# ground truth gathered by running many configurations of blocks and threadsPerBlock on CIMS servers
# with fixed number of elements for 20 iterations each and taking their mean stratified by configuration
# note: all GT configurations ran on the system with the best observed overall performance (cuda3 in this case)

gt = pd.read_csv('vaGT10m.csv')
gt = gt.groupby(['blocks', 'threadsPerBlock']).mean().reset_index().sort_values('GPU')
gt.head(10)

,blocks,threadsPerBlock,elements,GPU
44,512,200,10000000.0,0.026260
43,512,128,10000000.0,0.026281
78,4096,750,10000000.0,0.026447
27,128,512,10000000.0,0.026456
36,256,400,10000000.0,0.026561
50,1024,32,10000000.0,0.026613
17,64,512,10000000.0,0.026618
29,128,1024,10000000.0,0.026644
104,32768,200,10000000.0,0.026652
97,16384,512,10000000.0,0.026657
